In [2]:
import subprocess
import subprocess
import sys
subprocess.run('tar -cf - -C /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script . | tar -xf - -C /tmp', shell=True)
subprocess.run('chmod +x /tmp/triton/backends/nvidia/bin/ptxas', shell=True)
subprocess.run('chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell', shell=True)
sys.path.insert(0, '/tmp')
# subprocess.run('chmod +x /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell', shell=True)
# subprocess.run('chmod +x /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas', shell=True)
import polars as pl

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')

train.head()
import subprocess
import sys

subprocess.run('tar --no-same-permissions -cf - -C /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script . | tar -xf - -C /tmp', shell=True, check=False)
subprocess.run('chmod +x /tmp/triton/backends/nvidia/bin/ptxas', shell=True)
subprocess.run('chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell', shell=True)
sys.path.insert(0, '/tmp')

In [3]:
import subprocess
import sys

# Install from local datasets (no internet needed)
subprocess.run(
    "pip install -q --no-index --find-links /kaggle/input/nemotron-packages/packages "
    "unsloth trl peft transformers datasets accelerate bitsandbytes",
    shell=True
)
subprocess.run(
    "pip install -q /kaggle/input/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
    shell=True
)
subprocess.run(
    "pip install -q /kaggle/input/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
    shell=True
)

ERROR: Could not find a version that satisfies the requirement unsloth (from versions: none)
ERROR: No matching distribution found for unsloth
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'



CompletedProcess(args='pip install -q /kaggle/input/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl', returncode=1)

In [11]:
import subprocess
import sys

# subprocess.run('tar --no-same-permissions -cf - -C /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script . | tar -xf - -C /tmp', shell=True, check=False)
# subprocess.run('chmod +x /tmp/triton/backends/nvidia/bin/ptxas', shell=True, check=False)
# subprocess.run('chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell', shell=True, check=False)
# sys.path.insert(0, '/tmp')

import site
cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

import kagglehub
import mamba_ssm
import torch
import polars as pl
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("Model loaded successfully.")

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
train.head()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded successfully.


id,prompt,answer
str,str,str
"""00066667""","""In Alice's Wonderland, a secre…","""10010111"""
"""000b53cf""","""In Alice's Wonderland, a secre…","""01000011"""
"""00189f6a""","""In Alice's Wonderland, secret …","""cat imagines book"""
"""001b24c4""","""In Alice's Wonderland, numbers…","""XXXVIII"""
"""001c63cb""","""In Alice's Wonderland, secret …","""wizard creates secret"""


In [4]:
import site
import os
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TRITON_PTXAS_PATH'] = '/tmp/triton/backends/nvidia/bin/ptxas'

cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

import kagglehub
import mamba_ssm
import torch
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# Configuration
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32  # Can be set to a maximum of 32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("Model loaded successfully.")

# Initialize LoRA Adapter
print(f"Initializing LoRA adapter with rank={LORA_RANK}...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)


# # Apply LoRA to the model
# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()


# # YOUR CODE HERE
# # --------------
# # model.train() 
# # --------------

inputs = tokenizer("Hello", return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))# # Save Adapter
# print(f"Saving adapter to {OUTPUT_DIR}...")
# model.save_pretrained(OUTPUT_DIR)

/tmp/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

NemotronH requires an initialized `NemotronHHybridDynamicCache` to return a cache. None was provided, so no cache will be returned.


Model loaded successfully.
Initializing LoRA adapter with rank=32...
Hello, AI World!" to the console. This is the simplest program to verify your Rust installation and development environment.

### ✅ Step-by-Step Explanation

#### 1. **Create a New Rust Project**
Use Rust’s built-in package manager


In [5]:

inputs = tokenizer("do you know paris, what are the most beautiful place to visit", return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))

do you know paris, what are the most beautiful place to visit there ?" They want a list of most beautiful places in Paris. Provide answer in English presumably. Should be friendly. We can list attractions: Eiffel Tower, Louvre Museum, Notre-Dame, Montmartre, Sacré-Cœur, Champs


In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [ ]:
print('Done.')